In [6]:
import asyncio
import io
import json
import wave

import websockets
from IPython.display import Audio, display

In [7]:

FLOW_WS = "ws://47.29.24.150:8080/ws/tts"      # change
VOICE_ID = "simran"

In [8]:

def pcm_to_wav(pcm: bytes, sample_rate: int = 24000):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)      # int16
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    buf.seek(0)
    return buf


def parse_frame(frame: bytes):
    """
    Server sends:
        {json_metadata}<raw_audio_bytes>

    Returns:
        metadata, audio_bytes
    """
    if not frame or frame[0] != ord("{"):
        return {}, frame

    depth = 0
    for i, b in enumerate(frame):
        if b == ord("{"):
            depth += 1
        elif b == ord("}"):
            depth -= 1
            if depth == 0:
                return json.loads(frame[: i + 1]), frame[i + 1 :]

    return {}, frame


async def synthesize(text: str):
    pcm_chunks = []
    sample_rate = 24000

    async with websockets.connect(FLOW_WS) as ws:

        payload = {
            "type": "synthesize",
            "call_id": "demo",
            "text_id": "1",
            "text": text,
            "streaming": True,
            "voice_id": VOICE_ID,
        }

        await ws.send(json.dumps(payload))

        while True:
            msg = await ws.recv()

            # JSON messages
            if isinstance(msg, str):
                meta = json.loads(msg)

                if meta["type"] == "audio_done":
                    break

                if meta["type"] == "error":
                    raise RuntimeError(meta["error"])

                if "sample_rate" in meta:
                    sample_rate = meta["sample_rate"]

            # Binary audio
            else:
                meta, audio = parse_frame(msg)

                if "sample_rate" in meta:
                    sample_rate = meta["sample_rate"]

                pcm_chunks.append(audio)

                if meta.get("is_final", False):
                    break

    pcm = b"".join(pcm_chunks)
    wav = pcm_to_wav(pcm, sample_rate)

    return wav

In [ ]:
text = '''
नमस्ते में प्रिया बोल रही हूँ बजाज फाइनेंस से। यह कॉल आपके पेंडिंग लोन EMI के regarding है, जो की बारह सौ रुपए है, जो पांच जुलाई दो हज़ार छब्बीस को due है। आप इस्सका payment कब तक कर सकते है?
'''
wav = await synthesize(text)

# display(Audio(wav.read(), rate=24000))
with open("output.wav", "wb") as f:
    f.write(wav.read())

In [17]:
import asyncio
import io
import json
import time
import wave

import websockets
from IPython.display import Audio, display

# -------------------------------------------------------------------
# Config
# -------------------------------------------------------------------
# -------------------------------------------------------------------
# Helpers
# -------------------------------------------------------------------

def pcm_to_wav(pcm: bytes, sample_rate: int = 24000):
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)      # int16 PCM
        wf.setframerate(sample_rate)
        wf.writeframes(pcm)
    buf.seek(0)
    return buf


def parse_frame(frame: bytes):
    """
    Binary frame format:
        {json_metadata}<raw_pcm_audio>
    """
    if not frame or frame[0] != ord("{"):
        return {}, frame

    depth = 0
    for i, b in enumerate(frame):
        if b == ord("{"):
            depth += 1
        elif b == ord("}"):
            depth -= 1
            if depth == 0:
                return json.loads(frame[: i + 1]), frame[i + 1 :]

    return {}, frame


# -------------------------------------------------------------------
# WebSocket
# -------------------------------------------------------------------

async def connect():
    ws = await websockets.connect(FLOW_WS)
    print("✅ Connected")
    return ws


async def disconnect(ws):
    await ws.close()
    print("✅ Disconnected")


# -------------------------------------------------------------------
# TTS
# -------------------------------------------------------------------

async def synthesize(ws, text):
    pcm_chunks = []
    sample_rate = 24000

    payload = {
        "type": "synthesize",
        "call_id": "demo",
        "text_id": "1",
        "text": text,
        "streaming": True,
        "voice_id": VOICE_ID,
    }

    start = time.perf_counter()
    first_chunk_time = None
    previous_chunk_time = None
    ttsb_values = []

    await ws.send(json.dumps(payload))

    while True:
        msg = await ws.recv()

        # ---------------- JSON ----------------

        if isinstance(msg, str):
            meta = json.loads(msg)

            if meta["type"] == "error":
                raise RuntimeError(meta["error"])

            if meta["type"] == "audio_done":
                break

            if "sample_rate" in meta:
                sample_rate = meta["sample_rate"]

        # ---------------- Binary audio ----------------

        else:
            now = time.perf_counter()

            meta, audio = parse_frame(msg)

            if "sample_rate" in meta:
                sample_rate = meta["sample_rate"]

            # TTFB
            if first_chunk_time is None:
                first_chunk_time = now
                print(f"TTFB : {(first_chunk_time - start) * 1000:.2f} ms")

            # TTSB (gap between successive audio chunks)
            if previous_chunk_time is not None:
                gap = (now - previous_chunk_time) * 1000
                ttsb_values.append(gap)
                print(f"TTSB : {gap:.2f} ms")

            previous_chunk_time = now

            pcm_chunks.append(audio)

            if meta.get("is_final", False):
                break

    if ttsb_values:
        print()
        print(f"Average TTSB : {sum(ttsb_values)/len(ttsb_values):.2f} ms")
        print(f"Minimum TTSB : {min(ttsb_values):.2f} ms")
        print(f"Maximum TTSB : {max(ttsb_values):.2f} ms")

    pcm = b"".join(pcm_chunks)
    return pcm_to_wav(pcm, sample_rate)


# -------------------------------------------------------------------
# Example
# -------------------------------------------------------------------

async def main():
    ws = await connect()

    try:
        wav = await synthesize(
            ws,
            "नमस्ते में प्रिया बोल रही हूँ बजाज फाइनेंस से।"
        )

        # Save
        wav.seek(0)
        with open("output.wav", "wb") as f:
            f.write(wav.read())

        print("\nSaved as output.wav")

        # Play in notebook
        wav.seek(0)
        display(Audio(wav.read()))

    finally:
        await disconnect(ws)


await main()

✅ Connected
TTFB : 352.12 ms
TTSB : 183.20 ms
TTSB : 90.92 ms

Average TTSB : 137.06 ms
Minimum TTSB : 90.92 ms
Maximum TTSB : 183.20 ms

Saved as output.wav


✅ Disconnected
